In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:

import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

csv_path = "/kaggle/input/datasets/uciml/iris/Iris.csv"

print("Dataset file exists:", os.path.exists(csv_path))

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())


In [ ]:

# Remove the unnecessary ID column
df = df.drop(columns=["Id"], errors="ignore")

print("Dataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSpecies distribution:")
print(df["Species"].value_counts())

print("\nStatistical summary:")
display(df.describe())


In [ ]:

sns.pairplot(
    df,
    hue="Species",
    diag_kind="hist"
)

plt.suptitle("Iris Dataset - Exploratory Pairplot", y=1.02)
plt.show()


In [ ]:

# Input features
X = df.drop(columns=["Species"])

# Output label
y = df["Species"]

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget classes:")
print(y.unique())


In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully!")


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, average="weighted"
        ),
        "Recall": recall_score(
            y_test, y_pred, average="weighted"
        ),
        "F1-Score": f1_score(
            y_test, y_pred, average="weighted"
        )
    })

    print("\n", name)
    print(classification_report(y_test, y_pred))

results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4))


In [ ]:

from sklearn.metrics import confusion_matrix

class_names = sorted(y.unique())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=class_names
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )

    ax.set_title(name)
    ax.set_xlabel("Predicted Species")
    ax.set_ylabel("Actual Species")

plt.tight_layout()
plt.show()


In [ ]:

results_df.plot(
    kind="bar",
    figsize=(10, 6),
    ylim=(0, 1.05)
)

plt.title("Logistic Regression vs Decision Tree")
plt.ylabel("Score")
plt.xlabel("Classification Model")
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:

from sklearn.inspection import DecisionBoundaryDisplay

features_2d = ["PetalLengthCm", "PetalWidthCm"]
X_2d = df[features_2d]

X_train_2d, X_test_2d, y_train_2d, y_test_2d = (
    train_test_split(
        X_2d,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

boundary_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, boundary_models.items()):
    model.fit(X_train_2d, y_train_2d)

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_2d,
        response_method="predict",
        plot_method="pcolormesh",
        alpha=0.25,
        ax=ax
    )

    sns.scatterplot(
        data=df,
        x="PetalLengthCm",
        y="PetalWidthCm",
        hue="Species",
        ax=ax
    )

    ax.set_title(name + " Decision Boundary")
    ax.set_xlabel("Petal Length (cm)")
    ax.set_ylabel("Petal Width (cm)")

plt.tight_layout()
plt.show()